**Author**: Edward Chhun (83878475)    

**Description**: This is a KNN (K-Nearest Neighbors) implementation for checking if a diabetes patient is readmitted, readmitted before or after 30 days. Here documents the whole process from splitting data all the way to evaluating the KNN classifier.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

seed = 1234
np.random.seed(seed)

pd.set_option('display.max_columns', None)

## Add Split Logic
---
## Preprocessing Should Happen for both training and test datasets

In [2]:
df = pd.read_csv('clean_training_data.csv')
df.head()

,Unnamed: 0,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,readmitted,race_Asian,race_Caucasian,race_Hispanic,race_Other,payer_code_CH,payer_code_CM,payer_code_CP,payer_code_DM,payer_code_FR,payer_code_HM,payer_code_MC,payer_code_MD,payer_code_MP,payer_code_OG,payer_code_OT,payer_code_PO,payer_code_SI,payer_code_SP,payer_code_UN,payer_code_WC,medical_specialty_Emergency_CriticalCare,medical_specialty_InternalMedicine_Subspecialty,medical_specialty_Maternal_Pediatric,medical_specialty_Missing,medical_specialty_Other,medical_specialty_PrimaryCare,medical_specialty_Psych_Rehab_Support,medical_specialty_Surgery,diag_1_140–239,diag_1_240–279,diag_1_280–289,diag_1_290–319,diag_1_320–389,diag_1_390–459,diag_1_460–519,diag_1_520–579,diag_1_580–629,diag_1_630–679,diag_1_680–709,diag_1_710–739,diag_1_740–759,diag_1_780–799,diag_1_800–999,diag_1_E & V codes,diag_2_140–239,diag_2_240–279,diag_2_280–289,diag_2_290–319,diag_2_320–389,diag_2_390–459,diag_2_460–519,diag_2_520–579,diag_2_580–629,diag_2_630–679,diag_2_680–709,diag_2_710–739,diag_2_740–759,diag_2_780–799,diag_2_800–999,diag_2_E & V codes,diag_3_140–239,diag_3_240–279,diag_3_280–289,diag_3_290–319,diag_3_320–389,diag_3_390–459,diag_3_460–519,diag_3_520–579,diag_3_580–629,diag_3_630–679,diag_3_680–709,diag_3_710–739,diag_3_740–759,diag_3_780–799,diag_3_800–999,diag_3_E & V codes,admission_type_id_2.0,admission_type_id_3.0,admission_type_id_4.0,admission_type_id_7.0,discharge_disposition_id_2.0,discharge_disposition_id_3.0,discharge_disposition_id_4.0,discharge_disposition_id_5.0,discharge_disposition_id_6.0,discharge_disposition_id_7.0,discharge_disposition_id_8.0,discharge_disposition_id_9.0,discharge_disposition_id_10.0,discharge_disposition_id_11.0,discharge_disposition_id_12.0,discharge_disposition_id_13.0,discharge_disposition_id_14.0,discharge_disposition_id_15.0,discharge_disposition_id_16.0,discharge_disposition_id_17.0,discharge_disposition_id_19.0,discharge_disposition_id_20.0,discharge_disposition_id_22.0,discharge_disposition_id_23.0,discharge_disposition_id_24.0,discharge_disposition_id_27.0,discharge_disposition_id_28.0,admission_source_id_2.0,admission_source_id_3.0,admission_source_id_4.0,admission_source_id_5.0,admission_source_id_6.0,admission_source_id_7.0,admission_source_id_8.0,admission_source_id_10.0,admission_source_id_11.0,admission_source_id_13.0,admission_source_id_14.0,admission_source_id_22.0,admission_source_id_25.0
0,83148,6.0,1,34,0,10,6,0,0,9,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,0.0,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False
1,8067,6.0,2,39,0,11,0,0,0,4,0.0,0.0,2.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,Fals

## K-Nearest Neighbors

### Ideas before approaching this problem

### Pre Processing Step:
1. Our data has way too many features (154), our KNN model will suffer from the curse of dimensionality and will not give us an accurate model.   
1.a. We want to aim to around ~50 features
1.b. We can combat this by feature selection, to reduce high-dimensional data into a lower-dimensional space before doing our KNN algorithms

### Overall pipe-line:
1. Split data
3. Log transform numeric features
4. Scaling
5. Dimension Reduction and Train KNN
6. Finalize with tuning hyperparameters and evaluate
7. Summary and Conclusion

# `1. Split Data`

In [4]:
"""
Splitting Data
"""
X = df.drop(columns=["readmitted"])
y = df["readmitted"]
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=.9, test_size=.1, shuffle=True, random_state=seed)
X_train.shape

(82430, 141)

# `2. Log Transform`

In [7]:
"""
Log-transform numeric features
"""

# Separate numeric/binary/low cardinal features

# Numeric cols
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

# Binary cols
binary_cols = [
    col for col in numeric_cols
    if X_train[col].nunique() == 2
]

# Low cardinality cols
min_cardinality = 10
low_card_cols = [
    col for col in numeric_cols
    if X_train[col].nunique() <= min_cardinality
]

# Now we select the log-canditates, seperate binary and low card from the numeric cols
log_candidates = list(
    set(numeric_cols) - set(binary_cols) - set(low_card_cols)
)

print(len(log_candidates))
log_candidates

8


['number_diagnoses',
 'time_in_hospital',
 'number_inpatient',
 'num_medications',
 'num_lab_procedures',
 'number_outpatient',
 'Unnamed: 0',
 'number_emergency']

In [6]:
X_train[log_candidates].skew()

number_diagnoses      -0.873608
time_in_hospital       1.130245
number_inpatient       3.643107
num_medications        1.318178
num_lab_procedures    -0.240612
number_outpatient      8.883190
Unnamed: 0             0.005475
number_emergency      21.026283
dtype: float64

#### Since log transform is for rightly-skewed data

We only work with rightly-skewed data, so data such as:
 
- num_lab_procedures    -0.238096     
- number_diagnoses      -0.876364     

Since they are between -1 and 1, we can leave it as is, but we can change this by reflecting them and 
doing a log transform later if we need to

In [8]:
# Remove lightly and negatively skewed data from log_candidates
log_candidates.remove("num_lab_procedures")
log_candidates.remove("number_diagnoses")

In [10]:
# Start log transform 
X_train[log_candidates] = np.log1p(X_train[log_candidates])
X_test[log_candidates] = np.log1p(X_test[log_candidates])

In [11]:
# Check skewness
X_train[log_candidates].skew()

time_in_hospital     0.101811
number_inpatient     1.446880
num_medications     -0.489463
number_outpatient    2.746238
Unnamed: 0          -1.978214
number_emergency     3.646894
dtype: float64

We can see our values are less skewed.

# `4. Scaling`
Since K-NN is distance based, we want to standardize features

In [12]:
scaler = StandardScaler()

# Save this just to have a copy of columns
original_cols = X_train.columns

# Change into dataframe, because scaler changes it to a numpy
X_train = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=original_cols,
    index=X_train.index
)

X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=original_cols,
    index=X_test.index
)

In [13]:
X_train

,Unnamed: 0,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,race_Asian,race_Caucasian,race_Hispanic,race_Other,payer_code_CH,payer_code_CM,payer_code_CP,payer_code_DM,payer_code_FR,payer_code_HM,payer_code_MC,payer_code_MD,payer_code_MP,payer_code_OG,payer_code_OT,payer_code_PO,payer_code_SI,payer_code_SP,payer_code_UN,payer_code_WC,medical_specialty_Emergency_CriticalCare,medical_specialty_InternalMedicine_Subspecialty,medical_specialty_Maternal_Pediatric,medical_specialty_Missing,medical_specialty_Other,medical_specialty_PrimaryCare,medical_specialty_Psych_Rehab_Support,medical_specialty_Surgery,diag_1_140–239,diag_1_240–279,diag_1_280–289,diag_1_290–319,diag_1_320–389,diag_1_390–459,diag_1_460–519,diag_1_520–579,diag_1_580–629,diag_1_630–679,diag_1_680–709,diag_1_710–739,diag_1_740–759,diag_1_780–799,diag_1_800–999,diag_1_E & V codes,diag_2_140–239,diag_2_240–279,diag_2_280–289,diag_2_290–319,diag_2_320–389,diag_2_390–459,diag_2_460–519,diag_2_520–579,diag_2_580–629,diag_2_630–679,diag_2_680–709,diag_2_710–739,diag_2_740–759,diag_2_780–799,diag_2_800–999,diag_2_E & V codes,diag_3_140–239,diag_3_240–279,diag_3_280–289,diag_3_290–319,diag_3_320–389,diag_3_390–459,diag_3_460–519,diag_3_520–579,diag_3_580–629,diag_3_630–679,diag_3_680–709,diag_3_710–739,diag_3_740–759,diag_3_780–799,diag_3_800–999,diag_3_E & V codes,admission_type_id_2.0,admission_type_id_3.0,admission_type_id_4.0,admission_type_id_7.0,discharge_disposition_id_2.0,discharge_disposition_id_3.0,discharge_disposition_id_4.0,discharge_disposition_id_5.0,discharge_disposition_id_6.0,discharge_disposition_id_7.0,discharge_disposition_id_8.0,discharge_disposition_id_9.0,discharge_disposition_id_10.0,discharge_disposition_id_11.0,discharge_disposition_id_12.0,discharge_disposition_id_13.0,discharge_disposition_id_14.0,discharge_disposition_id_15.0,discharge_disposition_id_16.0,discharge_disposition_id_17.0,discharge_disposition_id_19.0,discharge_disposition_id_20.0,discharge_disposition_id_22.0,discharge_disposition_id_23.0,discharge_disposition_id_24.0,discharge_disposition_id_27.0,discharge_disposition_id_28.0,admission_source_id_2.0,admission_source_id_3.0,admission_source_id_4.0,admission_source_id_5.0,admission_source_id_6.0,admission_source_id_7.0,admission_source_id_8.0,admission_source_id_10.0,admission_source_id_11.0,admission_source_id_13.0,admission_source_id_14.0,admission_source_id_22.0,admission_source_id_25.0
71556,0.898948,-1.945579,0.124053,-1.023721,-0.783583,0.105219,2.151580,3.176880,-0.637603,-0.217250,-0.213438,-0.411991,1.851452,-0.113902,-0.080263,-0.208106,2.405847,-0.295406,-0.26451,3.711334,-0.053327,0.710564,-0.082159,1.080535,0.546631,-0.080138,0.546949,-0.142831,-0.123127,-0.038023,-0.138571,-0.160297,-0.072245,-0.003483,-0.255141,0.631357,-0.189174,-0.029154,-0.100854,-0.031939,-0.077568,-0.023885,-0.227107,-0.157002,-0.03754,3.526785,-0.320328,-0.114625,-0.011015,0.0,-1.563255,-0.116832,-0.303276,-0.187344,-0.355767,-0.103341,-0.150849,-0.110537,1.531993,-0.336961,-0.315702,-0.2306,-0.080062,-0.160337,-0.226041,-0.022308,-0.284492,-0.269978,-0.1285,-0.160655,-0.510688,-0.172092,-0.164508,-0.11304,-0.671986,-0.336492,-0.200806,-0.292119,-0.061738,-0.190922,-0.131301,-0.032691,4.584382,-0.155659,-0.159899,-0.138526,1.692700,-0.158376,-0.178018,-0.132917,-0.666627,-0.266835,-0.190649,-0.258095,-0.05415,-0.157408,-0.138253,-0.030973,-0.215772,-0.139794,-0.227048,-0.471169,-0.473954,-0.009852,-0.014362,-0.146199,-0.399170,-0.090184,-0.107921,-0.380815,-0.078747,-0.033427,-0.014779,-0.008532,-0.127718,-0.004926,-0.062231,-0.059828,-0.024388,-0.009852,-0.011553,-0.007789,-0.004926,-0.140962,-0.063399,-0.021757,-0.006966,-0.037051,-0.103341,-0.042554,-0.180684,-0.093078,-0.151521,0.755

# `5. Dimension Reduction with KNN pipeline`
We still need to select the best features and reduce our dimensions for our KNN to be performant    
We can use SelectKBest features     

A general rule of thumb is to select **k** = $ \sqrt{n} $ where n is # of features

But we can learn the best # of features and K for our Nearest Neighbors for our dataset using a `GridSearchCV` library from scikit-learn

Constructing a `Pipeline` to learn both.

In [14]:
X_train.shape

(82430, 141)

In [15]:
y_train.shape

(82430,)

In [17]:
pipe = Pipeline([
    ("var", VarianceThreshold(threshold=0.0)),
    ("select", SelectKBest(score_func=f_classif)),
    ("knn", KNeighborsClassifier()),
])

param_grid = {
    "select__k": list(range(15, 30, 2)),
    "knn__n_neighbors": [260, 270, 280, 290, 300],
}

grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV accuracy:", round(grid_search.best_score_, 4))

Best params: {'knn__n_neighbors': 270, 'select__k': 27}
Best CV accuracy: 0.579


In [18]:
best_model = grid_search.best_estimator_

# Columns kept after variance threshold
var_mask = best_model.named_steps["var"].get_support()
cols_after_var = X_train.columns[var_mask]

# Columns kept after SelectKBest
select_mask = best_model.named_steps["select"].get_support()
selected_features = cols_after_var[select_mask]

print("Number selected:", len(selected_features))
print(list(selected_features))

Number selected: 27
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'change', 'diabetesMed', 'medical_specialty_Maternal_Pediatric', 'medical_specialty_Surgery', 'diag_1_140–239', 'diag_1_460–519', 'diag_1_630–679', 'diag_2_630–679', 'admission_type_id_3.0', 'discharge_disposition_id_3.0', 'discharge_disposition_id_5.0', 'discharge_disposition_id_6.0', 'discharge_disposition_id_11.0', 'discharge_disposition_id_13.0', 'discharge_disposition_id_14.0', 'discharge_disposition_id_22.0', 'admission_source_id_4.0', 'admission_source_id_6.0', 'admission_source_id_7.0']


In [19]:
# Predict and evaluate
y_pred_test = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_test))

Test Accuracy: 0.5793208865596681


# 

# 7. `Summary and Conclusion`

1. My best accuracy was ~58% validation and ~58% test.
2. I tried `dropping near constant features` and `reverting it` to see if I improve my KNN model, which it did, but this is later taken care of by SelectKBest features.
3. Using `log transform` on continuous data only is important so other features wouldn't pull the model that much, this normalized the data significantly.
4. Methods to train the model is to remove all constant features using `VarianceThreshold`, `Selecting the K best features` to reduce the dimensionality for my KNN to perform better, and `finding the best amount of K` neighbors for the classifier to improve its performance
5. Im trying to find other ways to improve the KNN by choosing other `hyperparameters`        
5.a. For the paramters, it tends to land on `high K values` for the `nearest-neighbors`, this suggests that our data is noisy and a smoother decision boundary works best for our model. I am experimenting with the range `200 - 300` since it is what the `GridSearch` has found to be best so far.

The best KNN pipeline found by cross-validation was:
Best params: {'knn__n_neighbors': 270, 'select__k': 27}

Future Improvements:
1. I would compute with a larger set of hyperparameters to fine tune the model and achieve a better score.
2. Look for more sophisticated methods of imputing data
3. Compare bases with other models (Logistic Classifier, Deep Neural Networks, etc.)